## Unity Catalog User-Defined Functions with Databricks Agent

### Installing Utilities and Libraries

In [ ]:
%pip install \
    databricks-sdk==0.49.0 \
    openai-agents==0.22.0 \
    mcp==2.0.0 \
    databricks-mcp==0.9.2 \
    unitycatalog-ai[databricks]==0.4.0 \
    "mlflow>=3.1"

### Restart your Python Environment

In [ ]:
dbutils.library.restartPython()

### Create UC Schema to Store Functions

In [ ]:
%sql 
CREATE SCHEMA IF NOT EXISTS YOUR_UC_NAME_GOES_HERE.user_functions

### Register the User-Defined UC Function

In [ ]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

CATALOG = "YOUR_UC_NAME_GOES_HERE"
SCHEMA = "user_functions"

def get_current_weather(location: str) -> str:
    """
    Gets the current weather condition for a location.

    Args:
        location (string): The location, e.g. San Francisco, Mumbai etc.

    Returns:
        string: a string containing the weather information
    """
    
    import requests 
    
    response = requests.get(
        f"https://wttr.in/{location}",
        params={
            "format": "j1"
        }
    )

    response.raise_for_status()

    return str(response.json())

uc_function_client = DatabricksFunctionClient()

function_info = uc_function_client.create_python_function(
    func=get_current_weather,
    catalog=CATALOG,
    schema=SCHEMA,
    replace=True
)

### Test the Function

In [ ]:
result = uc_function_client.execute_function(
    function_name=f"{CATALOG}.{SCHEMA}.get_current_weather",
    parameters={"location": "Mumbai"}
)

print(result.value)

### Set up the Workspace Client

In [ ]:
from databricks.sdk import WorkspaceClient

# Get Databricks runtime authentication
w = WorkspaceClient()

headers = w.config.authenticate()
token = headers["Authorization"].replace("Bearer ", "")
workspace_host = w.config.host.rstrip("/")

### Create and Execute the Agent

In [ ]:
from agents import (
    Agent,
    Runner,
    AsyncOpenAI,
    OpenAIChatCompletionsModel,
    set_tracing_disabled
)

from agents.mcp import MCPServerStreamableHttp
from databricks.sdk import WorkspaceClient

openai_client = AsyncOpenAI(
    api_key=token,
    base_url=f"{workspace_host}/serving-endpoints"
)

model = OpenAIChatCompletionsModel(
    model="databricks-claude-sonnet-4-5",
    openai_client=openai_client
)

set_tracing_disabled(True)

mcp_server_url = (
    f"{workspace_host}/api/2.0/mcp/functions/"
    f"{CATALOG}/{SCHEMA}"
)


# Connect to UC function through MCP
async with MCPServerStreamableHttp(
    name="UC-Functions",
    params={
        "url": mcp_server_url,
        "headers": {
            "Authorization": f"Bearer {token}"
        },
        "timeout": 60
    }
) as uc_server:
    
    # Create OpenAI Agent
    agent = Agent(
        name="Weather-Agent",

        instructions=(
            "You are a helpful assistant. "
            "Use the available tools to retrieve weather information"
        ),

        model=model,

        mcp_servers=[uc_server]
    )

    # Run Agent
    result = await Runner.run(
        agent,
        "What is the weather like in San Francisco currently?"
    )

    print(result.final_output)